# Notebook 3: Manual Validation Sampling & Team Assignment
Loads the finalized Stage 2 dataset (`flying_drones_stage2_clean.parquet`), draws a reproducible random sample of 100 patents, splits them evenly across 5 team members (20 patents each), and exports an annotation spreadsheet. Includes a post-annotation evaluation cell to compute empirical corpus precision and 95% confidence intervals.

In [1]:
import numpy as np
import pandas as pd

In [2]:
# 1. Load finalized Stage 2 dataset
df_final = pd.read_parquet("flying_drones_stage2_clean.parquet")
print(f"Loaded finalized Stage 2 corpus: {len(df_final):,} patents")

Loaded finalized Stage 2 corpus: 17,689 patents


In [3]:
# 2. Draw reproducible random sample of 100 patents
SAMPLE_SIZE = 100
NUM_MEMBERS = 5
PER_MEMBER = SAMPLE_SIZE // NUM_MEMBERS

val_sample = df_final.sample(n=SAMPLE_SIZE, random_state=42).copy().reset_index(drop=True)

# Assign 20 patents to each of the 5 group members
members = ["Francesco", "Riym", "Michel", "Satine", "Enrico"]
members = [i for i in members]
val_sample.insert(0, "reviewer", np.repeat(members, PER_MEMBER))

# Add blank columns for manual annotation (1 = Relevant Flying Drone Tech, 0 = False Positive)
val_sample["is_relevant"] = ""
val_sample["review_notes"] = ""

cols_to_export = [
    "reviewer",
    "is_relevant",
    "review_notes",
    "appln_id",
    "docdb_family_id",
    "appln_filing_date",
    "appln_auth",
    "appln_title",
    "appln_abstract",
    "cpc_codes",
    "applicants",
    "sim_score_centroid"
]
val_sample = val_sample[cols_to_export]

# Export to Excel (with CSV fallback)
try:
    val_sample.to_excel("drone_validation_sample_100.xlsx", index=False)
    print("Saved 100-patent validation sheet to 'drone_validation_sample_100.xlsx'")
except ImportError:
    val_sample.to_csv("drone_validation_sample_100.csv", index=False)
    print("Saved 100-patent validation sheet to 'drone_validation_sample_100.csv'")

print("\nAssignment breakdown:")
print(val_sample['reviewer'].value_counts().sort_index())

Saved 100-patent validation sheet to 'drone_validation_sample_100.xlsx'

Assignment breakdown:
reviewer
Enrico       20
Francesco    20
Michel       20
Riym         20
Satine       20
Name: count, dtype: int64


## Post-Annotation Precision & 95% Confidence Interval Evaluation
Run the cell below once all 5 team members have filled in `is_relevant` (`1` or `0`) in the exported file.

In [ ]:
# 3. Load completed validation sheet and compute empirical precision + 95% Wilson CI
import math

completed_path = "drone_validation_sample_100.xlsx"  # Update if using .csv
labeled_df = pd.read_excel(completed_path) if completed_path.endswith(".xlsx") else pd.read_csv(completed_path)

labeled_df['is_relevant'] = pd.to_numeric(labeled_df['is_relevant'], errors='coerce')
valid_rows = labeled_df.dropna(subset=['is_relevant'])

n = len(valid_rows)
k = int(valid_rows['is_relevant'].sum())
p_hat = k / n

# 95% Wilson Score Confidence Interval
z = 1.96
denom = 1 + (z**2 / n)
center = (p_hat + (z**2 / (2 * n))) / denom
margin = (z * math.sqrt((p_hat * (1 - p_hat) + (z**2 / (4 * n))) / n)) / denom

print(f"Annotated Records: {n} / {len(labeled_df)}")
print(f"Empirical Precision: {p_hat * 100:.2f}% ({k}/{n} relevant)")
print(f"95% Wilson Confidence Interval: [{(center - margin) * 100:.2f}%, {(center + margin) * 100:.2f}%]")

print("\nPrecision by Reviewer:")
print(valid_rows.groupby('reviewer')['is_relevant'].agg(['count', 'sum', 'mean']))